# Keypoint Detection & Pose Estimation Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Gaussian heatmap target

In [ ]:
```python

import numpy as np

import torch

def gaussian_heatmap(size, cx, cy, sigma=2.0):

    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")

    return np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2)).astype(np.float32)

hm = gaussian_heatmap(64, 32, 32, sigma=2.0)

print(f"peak: {hm.max():.3f} at ({hm.argmax() % 64}, {hm.argmax() // 64})")

In [ ]:
```

Per-keypoint heatmaps stacked along a channel axis give the full target tensor.

### Step 2: Tiny keypoint head

A U-Net-style model that outputs K heatmap channels.

In [ ]:
```python

import torch.nn as nn

import torch.nn.functional as F

class TinyKeypointNet(nn.Module):

    def __init__(self, num_keypoints=4, base=16):

        super().__init__()

        self.down1 = nn.Sequential(nn.Conv2d(3, base, 3, 2, 1), nn.ReLU(inplace=True))

        self.down2 = nn.Sequential(nn.Conv2d(base, base * 2, 3, 2, 1), nn.ReLU(inplace=True))

        self.mid = nn.Sequential(nn.Conv2d(base * 2, base * 2, 3, 1, 1), nn.ReLU(inplace=True))

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)

        self.up2 = nn.ConvTranspose2d(base, num_keypoints, 2, 2)

    def forward(self, x):

        h1 = self.down1(x)

        h2 = self.down2(h1)

        h3 = self.mid(h2)

        u1 = self.up1(h3)

        return self.up2(u1)

In [ ]:
```

Input `(N, 3, H, W)`, output `(N, K, H, W)`. Loss is per-pixel MSE against Gaussian targets.

### Step 3: Inference — extract keypoint coordinates

In [ ]:
```python

def heatmap_to_coords(heatmaps):

    """

    heatmaps: (N, K, H, W)

    returns:  (N, K, 2) float coordinates in image pixels

    """

    N, K, H, W = heatmaps.shape

    hm = heatmaps.reshape(N, K, -1)

    idx = hm.argmax(dim=-1)

    ys = (idx // W).float()

    xs = (idx % W).float()

    return torch.stack([xs, ys], dim=-1)

coords = heatmap_to_coords(torch.randn(2, 4, 32, 32))

print(f"coords: {coords.shape}")  # (2, 4, 2)

In [ ]:
```

One line at inference. For sub-pixel refinement, interpolate around the argmax.

### Step 4: Synthetic keypoint dataset

Simple: draw four points on a white canvas and learn to predict them.

In [ ]:
```python

def make_synthetic_sample(size=64):

    img = np.ones((3, size, size), dtype=np.float32)

    rng = np.random.default_rng()

    kps = rng.integers(8, size - 8, size=(4, 2))

    for cx, cy in kps:

        img[:, cy - 2:cy + 2, cx - 2:cx + 2] = 0.0

    hms = np.stack([gaussian_heatmap(size, cx, cy) for cx, cy in kps])

    return img, hms, kps

In [ ]:
```

Easy enough for a tiny model to learn in a minute.

### Step 5: Training

In [ ]:
```python

model = TinyKeypointNet(num_keypoints=4)

opt = torch.optim.Adam(model.parameters(), lr=3e-3)

for step in range(200):

    batch = [make_synthetic_sample() for _ in range(16)]

    imgs = torch.from_numpy(np.stack([b[0] for b in batch]))

    hms = torch.from_numpy(np.stack([b[1] for b in batch]))

    pred = model(imgs)

    # Upsample pred to full resolution

    pred = F.interpolate(pred, size=hms.shape[-2:], mode="bilinear", align_corners=False)

    loss = F.mse_loss(pred, hms)

    opt.zero_grad(); loss.backward(); opt.step()

In [ ]:
```

## Exercises

In [ ]:
1. **(Easy)** Train the tiny keypoint model on the synthetic 4-point dataset. Report mean L2 error between predicted and true keypoints after 200 steps.
2. **(Medium)** Add sub-pixel refinement: given the argmax position, fit a 1D parabola along x and y from the neighbouring pixels. Report the accuracy gain vs integer argmax.
3. **(Hard)** Build a 2-person synthetic dataset where each image shows two instances of the 4-keypoint pattern. Train a bottom-up pipeline with PAFs that predict which keypoint belongs to which instance, and evaluate OKS.